#### **Paper information**

---

##### *Personal KV-Cache Retrieval - the memory/identity layer*

- The core idea is unusual and important: instead of repeatedly stuffing personal information into a long prompt, JARVIS converts personal evidence into reusable **KV-cache states**, retrieves only the relevant evidence, and attaches those states to the model during inference. The paper keeps the underlying model frozen.
- Its visual side goes further: it tries to isolate identity-breaing patches using subject masks, diffusion difficulty and CLIP relevance rather than following the model to rely on backgrounds.

---

##### *JUST A RATHER VERY INTELLIGENT SPOKEN AGENT - the interaction layer*

This paper tackles a different problem: even good long-running agents leave the user disconnected from what they are doing. Its JARVIS sits **between the human and worker agent**, watches execution traces, answers questions while work continues and asks the user for help at failure-sensitive checkpoints.

It defines to evaluation dimensions:
- **Agent collaboration**: does JARVIS improve the worker's task completion?
- **User interation**: does JRAVIS make the agent's ongoing work understandable?

Its preliminary experiment reported a Claude Opus 4.7 worker going from **64.01 → 75.79** Overall score with GPT-level JARVIS mediation.

---

##### *OpenJarvis - the system architecture*

This is the architectural backbone.
It decomposes JARVIS into:
```
Intelligence
Engine
Agents
Tools & Memory
Learning
```
These become independently configurate parts of a single typed specification.
Then it proposes.
```
Diagnose
   ↓
Propose modification
   ↓
Execute candidate
   ↓
Held-out evaluation
   ↓
Accept / Reject
   ↓
Repeat
```
The important idea is that **the model is not the entire AI system**. The surrounding runtime, tools, memory and agent loop must be optimnized with it.

---

In [1]:
import os, json, time, math, dataclasses, statistics, textwrap, re
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple, Callable
from collections import deque, defaultdict

import numpy as np

print("Python prototype initialized.")

Python prototype initialized.


##### *1. Personal KV-cache retrieval*

The paper's central idea is **evidence-as-KV**.

Instead of repeatedly rebuilding a long personalized prompt, it pre-fills a frozen model with compact concept evidence and stores the resulting layer-wise key/value states for reuse.

Its offline evidence staege creates a compact profiule containing a cononical name, cateogry, short caption and fingerprint attributes. Its visual pipeline then mines discriminative patches.

At query time, only relevant evidence is retrived and attached to the current decoding state.

In [ ]:
@dataclass
class ConceptEvidence:
    concept_id: str
    name: str
    category: str
    caption: str
    attributes: Dict[str, str] = field(default_factory=dict)
    source_images: List[str] = field(default_factory=list)

    def as_text(self) -> str:
        attrs = "; ".join(f"{k}: {v}" for k, v in self.attributes.items())
        return (
            f"concept: {self.name}\n"
            f"category: {self.category}\n"
            f"caption: {self.caption}\n"
            f"attributes: {attrs}"
        )